# 05c ML Fixed Embeddings No Log Target

This secondary notebook trains downstream regression models on the no-log embedding datasets generated by `03c_embeddings_v3_nolog.ipynb`.

Target: `systemic_risk_label`.

Experiments:

- GraphSAGE fixed 32-dim
- GraphSAGE fixed 64-dim
- Node2Vec fixed 32-dim
- Node2Vec fixed 64-dim

In [ ]:
from pathlib import Path
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]
TARGET_COL = "systemic_risk_label"
DISPLAY_COLS = [
    "dataset",
    "model",
    "train_mae",
    "validation_mae",
    "test_mae",
    "train_rmse",
    "validation_rmse",
    "test_rmse",
]

print(f"Project root: {PROJECT_ROOT}")

## Load No-Log Embedding Datasets

In [ ]:
DATASETS = {
    "graphsage_32": "graphsage_fixed_32_srisk_nolog_dataset.parquet",
    "graphsage_64": "graphsage_fixed_64_srisk_nolog_dataset.parquet",
    "node2vec_32": "node2vec_fixed_32_srisk_nolog_dataset.parquet",
    "node2vec_64": "node2vec_fixed_64_srisk_nolog_dataset.parquet",
}

loaded = {}
for name, filename in DATASETS.items():
    df, feature_cols = load_gnn_dataset(
        PROJECT_ROOT,
        target_col=TARGET_COL,
        filename=filename,
    )
    loaded[name] = (df, feature_cols)
    print(f"{name:14s} {df.shape}  embedding_cols={len(feature_cols)}")

In [ ]:
for name, (df, feature_cols) in loaded.items():
    print("
", name)
    print(df[["bank_id", "year", "quarter", "period", TARGET_COL]].head())

## Create Trainers

In [ ]:
trainers = {}
for name, (df, feature_cols) in loaded.items():
    trainers[name] = ModelTrainer(
        df=df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
    )
    trainer = trainers[name]
    print(f"{name:14s} train={trainer.train_df.shape} val={trainer.val_df.shape} test={trainer.test_df.shape}")

## Define Models

In [ ]:
candidate_models = {
    "linear_regression": make_pipeline(LinearRegression(), scale_features=True),
    "ridge": make_pipeline(Ridge(alpha=1.0), scale_features=True),
    "random_forest": make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "xgboost": make_pipeline(
        XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
        scale_features=True,
    ),
}

list(candidate_models)

## Train All No-Log Experiments

In [ ]:
leaderboards = []
for dataset_name, trainer in trainers.items():
    trainer.train_all(candidate_models)
    board = trainer.leaderboard().copy()
    board.insert(0, "dataset", dataset_name)
    leaderboards.append(board)

all_results = pd.concat(leaderboards, ignore_index=True)
all_results[DISPLAY_COLS].sort_values(["validation_rmse", "validation_mae"]).reset_index(drop=True)

## Best Model Per Dataset

In [ ]:
best_rows = []
for dataset_name, trainer in trainers.items():
    board = trainer.leaderboard().copy()
    board.insert(0, "dataset", dataset_name)
    best_rows.append(board.iloc[0])

best_results = pd.DataFrame(best_rows)
best_results[DISPLAY_COLS].sort_values(["validation_rmse", "validation_mae"]).reset_index(drop=True)

## Inspect Test Predictions

In [ ]:
for dataset_name, trainer in trainers.items():
    print("
", dataset_name, "best=", trainer.best_name())
    display(trainer.test_predictions().head(10))